# 02 · Silver: conform SAP data

Turns source-shaped bronze tables into typed, business-named silver tables. This is where the SAP
conventions are handled, once, so nothing downstream needs to know about them:

| SAP convention | Example | Silver rule |
|---|---|---|
| Dates as `YYYYMMDD`, `00000000` = empty | `EINDT = 20250314` | `DATE`, `NULL` when initial |
| Leading zeros (ALPHA conversion) | `LIFNR = 0000100023` | key kept as-is, display number without zeros |
| Debit/credit indicator instead of a sign | `SHKZG = H` on a 102 reversal | signed quantities and amounts |
| Price unit | `NETPR = 180.00` per `PEINH = 1000` | `net_unit_price = NETPR / PEINH` |
| Blank vs `X` flags | `LOEKZ = L`, `ZLSPR = R` | booleans |
| Inverted exchange-rate date | `GDATU = 79749168` | `valid_from = 99999999 − GDATU`, with `valid_to` |
| Indirect quotation stored as a negative rate | `UKURS = −1.17260` | `rate = 1 / |UKURS|` |
| Invoice ↔ accounting document link | `BKPF-AWKEY = BELNR‖GJAHR` | payment keyed by supplier invoice |

In [ ]:
TABLE_FORMAT = "delta"

In [ ]:
from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F


def c(name) -> Column:
    return F.col(name) if isinstance(name, str) else name


def sap_date(name) -> Column:
    value = c(name)
    return (F.when(value.isNull() | (value == "00000000"), F.lit(None).cast("date"))
             .otherwise(F.to_date(value, "yyyyMMdd")))


def text(name) -> Column:
    """Blank SAP values arrive as NULL from the extract; treat NULL and '' the same."""
    return F.coalesce(F.trim(c(name)), F.lit(""))


def flag(name) -> Column:
    return text(name) != ""


def amount(name) -> Column:
    return c(name).cast("decimal(18,2)")


def quantity(name) -> Column:
    return c(name).cast("decimal(18,3)")


def signed(value: Column, shkzg: str = "SHKZG") -> Column:
    return F.when(F.col(shkzg) == "H", -value).otherwise(value)


def alpha_out(name) -> Column:
    """SAP ALPHA conversion exit (output direction): drop leading zeros, keep a lone zero."""
    return F.regexp_replace(c(name), "^0+(?=.)", "")


def bronze(table: str, keys: list) -> DataFrame:
    """Latest ingested row per business key, so re-delivered rows never duplicate."""
    w = Window.partitionBy(*keys).orderBy(F.col("_ingested_at").desc())
    return (spark.table(f"bronze_{table}")
            .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn"))


def save(df: DataFrame, name: str):
    df.write.mode("overwrite").format(TABLE_FORMAT).option("overwriteSchema", "true").saveAsTable(name)
    print(f"{name:<40} {spark.table(name).count():>8,} rows")

## Master data

In [ ]:
save(bronze("lfa1", ["LIFNR"]).select(
    F.col("LIFNR").alias("supplier_id"),
    alpha_out("LIFNR").alias("supplier_number"),
    F.col("NAME1").alias("supplier_name"),
    F.col("LAND1").alias("country_code"),
    F.col("ORT01").alias("city"),
    F.col("KTOKK").alias("account_group"),
    flag("SPERR").alias("is_purchasing_blocked"),
    flag("LOEVM").alias("is_marked_for_deletion"),
    sap_date("ERDAT").alias("created_on"),
), "silver_supplier")

makt = bronze("makt", ["MATNR", "SPRAS"]).filter(F.col("SPRAS") == "E").select("MATNR", "MAKTX")
save(bronze("mara", ["MATNR"]).join(makt, "MATNR", "left").select(
    F.col("MATNR").alias("material_id"),
    alpha_out("MATNR").alias("material_number"),
    F.coalesce(F.col("MAKTX"), alpha_out("MATNR")).alias("material_description"),
    F.col("MTART").alias("material_type"),
    F.col("MATKL").alias("material_group"),
    F.col("MEINS").alias("base_unit"),
    sap_date("ERSDA").alias("created_on"),
    flag("LVORM").alias("is_marked_for_deletion"),
), "silver_material")

save(bronze("t023t", ["SPRAS", "MATKL"]).filter(F.col("SPRAS") == "E").select(
    F.col("MATKL").alias("material_group"),
    F.col("WGBEZ").alias("material_group_name"),
), "silver_material_group")

save(bronze("t001w", ["WERKS"]).select(
    F.col("WERKS").alias("plant"),
    F.col("NAME1").alias("plant_name"),
    F.col("ORT01").alias("city"),
    F.col("LAND1").alias("country_code"),
), "silver_plant")

save(bronze("t024", ["EKGRP"]).select(
    F.col("EKGRP").alias("purchasing_group"),
    F.col("EKNAM").alias("purchasing_group_name"),
), "silver_purchasing_group")

## Exchange rates
TCURR stores the validity date inverted and indirect quotations as negative numbers. Translation
ratios (TCURF) are 1:1 for EUR and USD in this extract; a production build would apply FFACT/TFACT too.

In [ ]:
by_currency = Window.partitionBy("from_currency", "to_currency").orderBy("valid_from")
fx = (bronze("tcurr", ["KURST", "FCURR", "TCURR", "GDATU"])
      .filter(F.col("KURST") == "M")
      .select(
          F.col("FCURR").alias("from_currency"),
          F.col("TCURR").alias("to_currency"),
          F.to_date((F.lit(99999999) - F.col("GDATU").cast("int")).cast("string"), "yyyyMMdd").alias("valid_from"),
          F.col("UKURS").cast("decimal(18,5)").alias("quoted_rate"))
      .withColumn("rate_to_local", F.round(
          F.when(F.col("quoted_rate") < 0, F.lit(1) / F.abs("quoted_rate")).otherwise(F.col("quoted_rate")), 6
      ).cast("decimal(18,6)"))
      .withColumn("valid_to", F.coalesce(F.date_sub(F.lead("valid_from").over(by_currency), 1),
                                         F.lit("9999-12-31").cast("date"))))
save(fx, "silver_fx_rate")

## Requisitions and purchase orders

In [ ]:
save(bronze("eban", ["BANFN", "BNFPO"]).select(
    F.col("BANFN").alias("purchase_requisition"),
    F.col("BNFPO").alias("purchase_requisition_item"),
    F.col("BSART").alias("requisition_type"),
    flag("LOEKZ").alias("is_deleted"),
    F.col("STATU").alias("processing_status"),
    F.col("EKGRP").alias("purchasing_group"),
    F.col("AFNAM").alias("requisitioner"),
    F.col("TXZ01").alias("item_text"),
    text("MATNR").alias("material_id"),
    F.col("WERKS").alias("plant"),
    F.col("MATKL").alias("material_group"),
    quantity("MENGE").alias("quantity"),
    F.col("MEINS").alias("unit"),
    sap_date("BADAT").alias("requisition_date"),
    sap_date("FRGDT").alias("release_date"),
    sap_date("LFDAT").alias("requested_delivery_date"),
    (amount("PREIS") / F.col("PEINH").cast("int")).cast("decimal(18,6)").alias("valuation_unit_price"),
    F.col("WAERS").alias("currency"),
    text("KNTTP").alias("account_assignment_category"),
    text("EBELN").alias("purchase_order"),
    F.col("EBELP").alias("purchase_order_item"),
), "silver_purchase_requisition_item")

save(bronze("ekko", ["EBELN"]).select(
    F.col("EBELN").alias("purchase_order"),
    F.col("BUKRS").alias("company_code"),
    F.col("BSART").alias("purchase_order_type"),
    F.col("LIFNR").alias("supplier_id"),
    F.col("EKORG").alias("purchasing_organization"),
    F.col("EKGRP").alias("purchasing_group"),
    F.col("WAERS").alias("currency"),
    sap_date("BEDAT").alias("po_date"),
    sap_date("AEDAT").alias("created_on"),
    F.col("ERNAM").alias("created_by"),
    F.col("ZTERM").alias("payment_terms"),
    F.col("FRGKE").alias("release_indicator"),
), "silver_purchase_order")

save(bronze("ekpo", ["EBELN", "EBELP"]).select(
    F.col("EBELN").alias("purchase_order"),
    F.col("EBELP").alias("purchase_order_item"),
    (text("LOEKZ") == "L").alias("is_deleted"),
    F.col("TXZ01").alias("item_text"),
    text("MATNR").alias("material_id"),
    F.col("WERKS").alias("plant"),
    text("LGORT").alias("storage_location"),
    F.col("MATKL").alias("material_group"),
    quantity("MENGE").alias("order_quantity"),
    F.col("MEINS").alias("order_unit"),
    amount("NETPR").alias("net_price"),
    F.col("PEINH").cast("int").alias("price_unit"),
    (amount("NETPR") / F.col("PEINH").cast("int")).cast("decimal(18,6)").alias("net_unit_price"),
    amount("NETWR").alias("net_amount"),
    text("MWSKZ").alias("tax_code"),
    text("KNTTP").alias("account_assignment_category"),
    flag("ELIKZ").alias("is_delivery_completed"),
    flag("EREKZ").alias("is_final_invoice"),
    flag("WEBRE").alias("is_gr_based_invoice_verification"),
    text("BANFN").alias("purchase_requisition"),
    F.col("BNFPO").alias("purchase_requisition_item"),
), "silver_purchase_order_item")

save(bronze("eket", ["EBELN", "EBELP", "ETENR"]).select(
    F.col("EBELN").alias("purchase_order"),
    F.col("EBELP").alias("purchase_order_item"),
    F.col("ETENR").alias("schedule_line"),
    sap_date("EINDT").alias("delivery_date"),
    sap_date("SLFDT").alias("statistical_delivery_date"),
    quantity("MENGE").alias("scheduled_quantity"),
    quantity("WEMNG").alias("delivered_quantity"),
), "silver_purchase_order_schedule_line")

## Purchase order history (EKBE)
Goods receipts (`VGABE = 1`) and invoice receipts (`VGABE = 2`) per PO item. `AREWR` is the GR/IR
clearing value in local currency, which is what the GR/IR balance is built from.

In [ ]:
save(bronze("ekbe", ["EBELN", "EBELP", "ZEKKN", "VGABE", "GJAHR", "BELNR", "BUZEI"]).select(
    F.col("EBELN").alias("purchase_order"),
    F.col("EBELP").alias("purchase_order_item"),
    F.col("ZEKKN").alias("account_assignment_number"),
    F.col("VGABE").alias("event_type"),
    F.when(F.col("VGABE") == "1", "Goods receipt")
     .when(F.col("VGABE") == "2", "Invoice receipt").otherwise("Other").alias("event_type_name"),
    F.col("GJAHR").alias("document_year"),
    F.col("BELNR").alias("document_number"),
    F.col("BUZEI").alias("document_item"),
    text("BWART").alias("movement_type"),
    sap_date("BUDAT").alias("posting_date"),
    signed(quantity("MENGE")).alias("quantity"),
    signed(amount("DMBTR")).alias("amount_local"),
    signed(amount("WRBTR")).alias("amount_document"),
    signed(amount("AREWR")).alias("grir_clearing_value_local"),
    F.col("WAERS").alias("document_currency"),
    F.col("HSWAE").alias("local_currency"),
    F.col("LFBNR").alias("reference_document"),
    F.col("LFGJA").alias("reference_document_year"),
), "silver_purchase_order_history")

## Supplier invoices and payments
The supplier's own invoice number (`XBLNR`) is keyed in by hand, so the same invoice can arrive as
`INV-004512`, `INV4512` or `4512`. A normalised reference is kept for duplicate detection.

In [ ]:
reference_digits = F.regexp_replace("XBLNR", "[^0-9]", "")
save(bronze("rbkp", ["BELNR", "GJAHR"]).select(
    F.concat("BELNR", "GJAHR").alias("invoice_key"),
    F.col("BELNR").alias("supplier_invoice"),
    F.col("GJAHR").alias("fiscal_year"),
    F.col("BUKRS").alias("company_code"),
    F.col("LIFNR").alias("supplier_id"),
    sap_date("BLDAT").alias("document_date"),
    sap_date("BUDAT").alias("posting_date"),
    F.col("XBLNR").alias("supplier_reference"),
    F.when(reference_digits != "", F.regexp_replace(reference_digits, "^0+", ""))
     .otherwise(F.upper(F.regexp_replace("XBLNR", "[^A-Za-z0-9]", ""))).alias("supplier_reference_normalised"),
    F.col("WAERS").alias("currency"),
    amount("RMWWR").alias("gross_amount"),
    amount("WMWST1").alias("tax_amount"),
    (amount("RMWWR") - amount("WMWST1")).alias("net_amount"),
    F.col("ZTERM").alias("payment_terms"),
    sap_date("ZFBDT").alias("baseline_date"),
    F.col("ZBD1T").cast("int").alias("net_payment_days"),
    F.date_add(sap_date("ZFBDT"), F.col("ZBD1T").cast("int")).alias("due_date"),
    flag("ZLSPR").alias("is_payment_blocked"),
    text("ZLSPR").alias("payment_block_reason"),
    F.col("RBSTAT").alias("invoice_status"),
    F.col("USNAM").alias("entered_by"),
), "silver_supplier_invoice")

save(bronze("rseg", ["BELNR", "GJAHR", "BUZEI"]).select(
    F.concat("BELNR", "GJAHR").alias("invoice_key"),
    F.col("BELNR").alias("supplier_invoice"),
    F.col("GJAHR").alias("fiscal_year"),
    F.col("BUZEI").alias("invoice_item"),
    F.col("EBELN").alias("purchase_order"),
    F.col("EBELP").alias("purchase_order_item"),
    text("MATNR").alias("material_id"),
    F.col("WERKS").alias("plant"),
    signed(amount("WRBTR")).alias("amount_document"),
    signed(quantity("MENGE")).alias("quantity"),
    F.col("BSTME").alias("order_unit"),
    text("MWSKZ").alias("tax_code"),
    F.col("LFBNR").alias("goods_receipt_document"),
    F.col("LFGJA").alias("goods_receipt_year"),
), "silver_supplier_invoice_item")

invoice_postings = (bronze("bkpf", ["BUKRS", "BELNR", "GJAHR"])
                    .filter(F.col("AWTYP") == "RMRP")
                    .select("BUKRS", "BELNR", "GJAHR", "AWKEY"))
save(bronze("bsak", ["BUKRS", "BELNR", "GJAHR", "BUZEI"])
     .join(invoice_postings, ["BUKRS", "BELNR", "GJAHR"])
     .select(
         F.concat(F.substring("AWKEY", 1, 10), F.substring("AWKEY", 11, 4)).alias("invoice_key"),
         F.substring("AWKEY", 1, 10).alias("supplier_invoice"),
         F.substring("AWKEY", 11, 4).alias("fiscal_year"),
         F.col("BUKRS").alias("company_code"),
         F.col("BELNR").alias("accounting_document"),
         F.col("GJAHR").alias("accounting_document_year"),
         F.col("BUZEI").alias("accounting_document_item"),
         F.col("LIFNR").alias("supplier_id"),
         sap_date("AUGDT").alias("clearing_date"),
         F.col("AUGBL").alias("clearing_document"),
         amount("DMBTR").alias("cleared_amount_local"),
         amount("WRBTR").alias("cleared_amount_document"),
         F.col("WAERS").alias("currency"),
     ), "silver_supplier_invoice_payment")